# IT 630 · Friday, October 30: Put the table on a map

**50-minute guided walkthrough · Google Colab · individual participation**

A community newsletter wants a map of New York City's five boroughs.
Its editor asks: **Which borough has the most residents, and is it also the
most densely populated?** You will load boundaries, attach a population table,
make maps, and explain what those maps can and cannot tell the editor.

You have used tables and joins. GeoPandas adds a **geometry** column: the shape
that belongs to each row. Today you will practice the GeoPandas workflow from
Wednesday; all necessary code and definitions are here, even if you missed it.
No previous notebook or GIS experience is required.

**Start:** Choose **File → Save a copy in Drive** and rename your copy
`IT630_Geo_YourName`. Run code cells in order with Shift+Enter. Double-click
an **Answer** text cell to write. Keep the outputs. You may use AI for syntax
help, but check its code against the stated observable result and explain the
result yourself. You do not need an answer key.

| Minutes | Work |
|---|---|
| 0–5 | Save your copy and run setup |
| 5–13 | Load and inspect the geographic table (Q1) |
| 13–22 | Merge the population table (Q2) |
| 22–32 | Calculate density and compare maps (Q3) |
| 32–42 | Change a question and investigate a missing match (Q4–Q5) |
| 42–50 | Write a data note and save your work (Q6–Q7) |

A code cell produces evidence; your short answers explain it. If you get
stuck, use the hints below the questions. After two minutes on an error,
record the cell and error in your answer and continue wherever possible.


## 1. Setup · 5 minutes

Run this cell once and wait for it to finish. It installs Python packages
inside this Colab session, not on your laptop. A CPU runtime is sufficient.
Do not choose a GPU. If Colab asks you to restart the session after the install,
restart it, then continue with the imports below. If a later cell reports an
undefined name, rerun from the imports down; a disconnected session loses variables.


In [ ]:
%pip install -q geopandas==1.2.0 pandas==2.2.3 matplotlib==3.10.0 numpy==2.1.3 shapely==2.2.0 pyproj==3.8.0 pyogrio==0.13.0


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from io import StringIO
from pathlib import Path
from urllib.request import urlopen
from hashlib import sha256

print("GeoPandas:", gpd.__version__, "| pandas:", pd.__version__)


**Observable result:** GeoPandas `1.2.0` and pandas `2.2.3` print without an error.

## 2. Load the geographic table · 8 minutes

The **NYC Department of City Planning** supplied these borough boundaries,
clipped to the shoreline at mean high tide, for **2016**. We use the fixed
`nybb_16a` copy preserved by the GeoPandas geodatasets project.
[Dataset description](https://geodatasets.readthedocs.io/en/latest/introduction.html#ny)
· [fixed archive](https://raw.githubusercontent.com/geopandas/geodatasets/60617613c578791a449b60230083c39f207711fa/data_backup/nybb_16a.zip).

The supplied download code checks a **checksum**, a fingerprint of the file,
so everyone analyzes the same bytes. You do not need to write download code.
`read_file` reads the map data inside the ZIP into a **GeoDataFrame**: a table
with a geometry column and a coordinate reference system (**CRS**).
A CRS tells us what the coordinates and their units mean.


In [ ]:
BOUNDARY_URL = "https://raw.githubusercontent.com/geopandas/geodatasets/60617613c578791a449b60230083c39f207711fa/data_backup/nybb_16a.zip"
BOUNDARY_SHA256 = "a303be17630990455eb079777a6b31980549e9096d66d41ce0110761a7e2f92a"
with urlopen(BOUNDARY_URL, timeout=60) as response:
    boundary_bytes = response.read()
assert sha256(boundary_bytes).hexdigest() == BOUNDARY_SHA256, "Download differs from the class specimen; stop and report it."
boundary_file = Path("nybb_16a.zip")
boundary_file.write_bytes(boundary_bytes)
boroughs = gpd.read_file("zip://" + str(boundary_file.resolve()) + "!nybb_16a/nybb.shp")
boroughs = boroughs[["BoroCode", "BoroName", "geometry"]].sort_values("BoroCode").reset_index(drop=True)
print("Rows:", len(boroughs))
print("CRS:", boroughs.crs)
print("Coordinate unit:", boroughs.crs.axis_info[0].unit_name)
display(boroughs)
print("Geometry types:", boroughs.geom_type.tolist())


**Observable result:** five rows, one for each borough; CRS `EPSG:2263`;
coordinate unit `US survey foot`; geometry type `MultiPolygon` in all five rows.
A **polygon** is an area enclosed by a boundary. A **MultiPolygon** groups
several such pieces into one shape, for example a mainland and islands.
Five rows need not mean five separate pieces on a map.

**Q1.** What does one row represent? What extra information does `geometry`
hold that `BoroName` alone cannot provide? Answer in two sentences.

If the download fails with a timeout or network error, rerun that cell once.
A checksum failure is different: keep the error and ask for help rather than
removing the check.


**Answer Q1:**

Write here.


## 3. Attach a population table · 9 minutes

The five counts below are transcribed from the US Census Bureau's
[January 2023 NYC history article](https://www.census.gov/about/history/stories/monthly/2023/january-2023.html).
They are **2020 Census counts**, not current estimates. The small CSV is
included in this notebook so you do not need a Census account or API key.
We added the borough codes to match the boundary file: 1 Manhattan, 2 Bronx,
3 Brooklyn, 4 Queens, 5 Staten Island. The codes identify places, not amounts.

Run the cell and inspect both tables before the merge.


In [ ]:
population_csv = """BoroCode,borough,population_2020
1,Manhattan,1694251
2,Bronx,1472654
3,Brooklyn,2736074
4,Queens,2405464
5,Staten Island,495747
"""
population = pd.read_csv(StringIO(population_csv))
display(population)
print("Boundary codes unique:", boroughs["BoroCode"].is_unique)
print("Population codes unique:", population["BoroCode"].is_unique)


This is an **attribute merge**, the same idea as joining ordinary tables on a
key. It matches equal `BoroCode` values; it does not calculate which shapes
intersect. `how="left"` keeps every boundary row.
`validate="one_to_one"` requires unique keys on both sides.
`indicator=True` adds `_merge`, which tells us whether each row found a match.
Uniqueness alone does **not** promise that every key will match.
[GeoPandas merge reference](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html).


In [ ]:
joined = boroughs.merge(population, on="BoroCode", how="left", validate="one_to_one", indicator=True)
display(joined[["BoroCode", "BoroName", "population_2020", "_merge"]])
print("Rows after merge:", len(joined))
print("Unmatched boundaries:", (joined["_merge"] != "both").sum())
print("Population total:", joined["population_2020"].sum())
assert len(joined) == len(boroughs)
assert joined["population_2020"].notna().all(), "A borough is missing its population."


**Observable result:** five output rows, all marked `both`, zero unmatched
boundaries. The population total is a count of people across all five boroughs.

**Q2.** Record the total. Explain why checking both the output row count and
`_merge` is better than checking the row count alone. Name the merge key.

*Hint:* A left merge keeps a boundary even if no population row matches it.


**Answer Q2:**

Write here.


## 4. Calculate a denominator, then map · 10 minutes

Population answers **how many residents**. Density answers **how many
residents per unit of area**. We need both a numerator and a denominator:

**people per square mile = population ÷ area in square miles**

These geometries use a local projected CRS, EPSG:2263, whose coordinates
are in US survey feet. GeoPandas `.area` therefore gives square survey feet.
Divide by `5280 ** 2` to express the area in square survey miles (the distinction
from international miles is negligible at our displayed precision).
The exponent `** 2` matters: we are converting area, not length.
Do not use this feet conversion on a file whose coordinates are longitude and
latitude in degrees. [Area and CRS reference](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.area.html).

**Limit:** this is an approximate teaching density using 2020 counts and
2016 shoreline-clipped shapes. It is not an official 2020 land-area density,
a current estimate, or a measurement of every neighborhood in a borough.


In [ ]:
assert joined.crs.to_epsg() == 2263, "The area conversion assumes EPSG:2263."
FEET_PER_MILE = 5280
joined["area_sq_miles"] = joined.geometry.area / FEET_PER_MILE ** 2
joined["people_per_sq_mile"] = joined["population_2020"] / joined["area_sq_miles"]
summary = joined[["BoroName", "population_2020", "area_sq_miles", "people_per_sq_mile"]]
display(summary.round({"area_sq_miles": 2, "people_per_sq_mile": 0}))


The next cell makes two **choropleth maps**: each borough's fill color represents
a numeric value. `column` chooses that value, `legend=True` shows its scale,
and `edgecolor` makes boundaries visible. The labeling loop is supplied;
you do not need to write it. [Mapping reference](https://geopandas.org/en/stable/docs/user_guide/mapping.html).

Before running, use the table to predict which borough will be darkest on each
map. Read each legend separately: the two panels have different units.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, column, title, units in [
    (axes[0], "population_2020", "2020 Census population", "Residents"),
    (axes[1], "people_per_sq_mile", "Approximate population density", "Residents per square mile"),
]:
    joined.plot(column=column, cmap="YlGnBu", edgecolor="white", linewidth=0.7,
                legend=True, legend_kwds={"label": units, "shrink": 0.65, "format": StrMethodFormatter("{x:,.0f}")}, ax=ax)
    for _, row in joined.iterrows():
        point = row.geometry.representative_point()
        ax.annotate(row["BoroName"], (point.x, point.y), ha="center", fontsize=9,
                    bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"})
    ax.set_title(title)
    ax.set_axis_off()
fig.suptitle("New York City boroughs · 2020 counts / 2016 boundary geometry", fontsize=13)
plt.tight_layout()
plt.show()


**Observable result:** two labeled maps of the same five boroughs, with a
numeric legend beside each. Darker blue means a higher value. The darkest
borough need not be the same in both panels. Use the table for exact numbers.

**Q3.** Name the borough with the most residents and the borough with the
highest approximate density. Give each value (nearest whole person or
person per square mile). Explain in one sentence why the rankings differ.

*Hint:* Compare a count with a count divided by an area. The largest colored
shape is not necessarily the largest value.


**Answer Q3:**

Write here.


## 5. Change the question, then test a missing match · 10 minutes

**Your code change.** The editor first wants boroughs with **at least 2,000,000**
residents. Run the next cell and record its names. Then change only
`minimum_population` to `1_500_000` and rerun. Underscores make large numbers
easier to read; Python treats `2_000_000` as `2000000`.
`>=` means **greater than or equal to**. `.loc` selects rows satisfying the
condition and the columns in the following list.


In [ ]:
minimum_population = 2_000_000  # After the first run, change this to 1_500_000.
selected = joined.loc[joined["population_2020"] >= minimum_population,
                      ["BoroName", "population_2020"]].sort_values("population_2020", ascending=False)
display(selected)
print("Selected boroughs:", len(selected))


**Q4.** List the selected boroughs at both thresholds. Which borough is added,
and why? Does this code change the original `joined` table?

*Hint:* Compare the added borough's count with both thresholds. The code stores
the selected rows in a new variable.


**Answer Q4:**

At 2,000,000:

At 1,500,000:

Explanation:


**A changed input.** Suppose the population file arrives without the row for
borough code 5. Predict what a left merge will do before running the supplied
experiment. It uses new variables, so the complete `joined` table stays intact.
The missing row is deliberately removed for this experiment; it is not missing
from the source.


**Answer Q5 — prediction (write before running):**

How many rows will the left merge return? What will appear in the unmatched
population cell?


In [ ]:
incomplete_population = population.loc[population["BoroCode"] != 5].copy()
incomplete_join = boroughs.merge(incomplete_population, on="BoroCode", how="left",
                                validate="one_to_one", indicator=True)
display(incomplete_join[["BoroName", "population_2020", "_merge"]])
print("Rows:", len(incomplete_join))
print("Missing population values:", incomplete_join["population_2020"].isna().sum())
print("Sum of available values:", incomplete_join["population_2020"].sum())


**Q5 — explanation.** Compare the result with your prediction. Identify the
unmatched borough and the amount by which the sum fell. Would replacing the
missing value with zero repair the data? Explain. The sum printed above skips
missing values; it is **not** a complete city total.

*Hint:* `NaN` means the population value is missing here. Zero would assert a
measured count of no residents. Compare with the complete table in Section 3.


**Answer Q5 — explanation:**

Write here.


## 6. Hand the editor a usable result · 8 minutes

A map needs a short **data note** so the next person knows what it supports.
This builds on the provenance record from the API and scraping walkthroughs.

**Q6.** Complete the note below in your own words. Include both source names
and links from Sections 2–3; the boundary year and population year; what one
row represents; the merge key and evidence of complete matching; and **one
claim this table does not support**. You are documenting your result, not
writing a general essay about maps.


**Answer Q6 — data note:**

- Intended use:
- Sources and years:
- One row represents:
- Merge key and completeness evidence:
- Area units and density limitation:
- One claim this table does not support:


**Q7.** The editor now asks: “Which neighborhood should get a new clinic next
year?” Can these tables answer that question? State one reason and identify
one additional dataset you would need. A borough is a larger area containing
many neighborhoods.

*Hint:* Consider the level of detail, the dates, and whether resident count
alone measures unmet healthcare need.


**Answer Q7:**

Write here.


## Save and submit

Check that Q1–Q7 contain your own responses, both map panels are visible,
and Q4 records both thresholds. Keep the changed threshold at `1_500_000`.
If a step failed, leave its error and describe what you tried.

Wait for Colab to save (or choose **File → Save**). Click **Share**, set
**General access → Anyone with the link**, keep the role **Viewer**, and copy
the URL. Paste your completed notebook's URL into this Canvas activity by the
Canvas deadline. Submit your own saved copy, not the starter link. You do not
need to upload a separate image or install anything on your laptop.
